# 여행시기특성 + 여행비용지출특성 (travel_timing_expenditure_2025.csv)

## [1] 데이터 불러오기 및 기본 정보 확인

### 데이터 불러오기 및 기본 정보 확인

In [1]:
import pandas as pd 

from utils.paths import set_project_root

set_project_root()

PosixPath('/Users/ryukyunghye/Documents/Github/out-traveler')

In [2]:
df = pd.read_csv('data/processed/travel_timing_expenditure_2025.csv')
print(df.shape)

(14342, 38)


### 중복 컬럼 제거

In [3]:
# '_x' 접미사를 가진 컬럼 중 대응하는 '_y'가 실제로 존재하는지 확인
x_cols = [col for col in df.columns if col.endswith('_x')]
same_pair_list = [col[:-2] for col in x_cols if col[:-2] + '_y' in df.columns]
print(f'x/y쌍: {len(same_pair_list)}') 


# 내용까지 전부 일치하는 쌍만 제거
full_duplicate = []

for name in same_pair_list:
    match_rate = (df[name+'_x'].fillna('__NA__')  == df[name+'_y'].fillna('__NA__')).mean()
    if match_rate == 1.0:
        full_duplicate.append(name)
    else:
        print(f'확인 필요: {name} [{match_rate}]')

drop_y = [name+'_y' for name in full_duplicate]
df = df.drop(columns=drop_y).rename(
    columns={col: col[:-2] for col in df.columns if col.endswith('_x') and col[:-2] in full_duplicate}
)

print(df.shape)

x/y쌍: 11
(14342, 27)


### 컬럼 정의표

| 컬럼영문명 | 컬럼한글명 | 데이터타입 | 사용여부 |
| --- | --- | --- | --- |
| RESPOND_ID | 응답자ID | int64 | ✓ |
| EXAMIN_BEGIN_DE | 조사시작일자 | int64 | ✓ |
| TOUR_CTPRVN_NM | 여행시도명 | str | ✓ |
| TOUR_SIGNGU_NM | 여행시군구명 | str | |
| TOUR_BEGIN_DE | 여행시작일자 | int64 | ✓ |
| TOUR_PD_VALUE | 여행기간값 | str | ✓ |
| TOUR_COM_NMPR_NM | 여행동행자인원명 | str | ✓ |
| COM_ONE_TY | 동행자1유형 | str | ✓ |
| COM_TWO_TY | 동행자2유형 | str | |
| COM_THREE_TY | 동행자3유형 | str | |
| COM_FOUR_TY | 동행자4유형 | str | |
| COM_FIVE_TY | 동행자5유형 | str | |
| COM_SIX_TY | 동행자6유형 | float64 | |
| TOUR_PURPS_NM | 여행목적명 | str | ✓ |
| SEXDSTN_FLAG_CD | 성별구분코드 | str | ✓ (보조) |
| AGRDE_FLAG_NM | 연령대구분명 | str | ✓ |
| MRRG_AT_NM | 결혼여부명 | str | ✓ (보조) |
| CHLDRN_TY_NM | 자녀유형명 | str | ✓ (보조) |
| OCCP_NM | 직업명 | str | ✓ (보조) |
| HSHLD_INCOME_DGREE_NM | 가구소득정도명 | str | |
| TOUR_TOT_CT_VALUE | 여행총비용값 | str | ✓ |
| TOUR_LDGMNT_CT_VALUE | 여행숙박비용값 | str | ✓ |
| TOUR_FOOD_CT_VALUE | 여행음식비용값 | str | ✓ |
| TOUR_TRNSPORT_CT_VALUE | 여행교통비용값 | str | ✓ |
| TOUR_SHOPNG_CT_VALUE | 여행쇼핑비용값 | str | ✓ |
| TOUR_ACTVTY_CT_VALUE | 여행액티비티비용값 | str | ✓ |
| TOUR_ETC_CT_VALUE | 여행기타비용값 | str | ✓ |

##### 사용 여부 판단 근거
- `TOUR_CTPRVN_NM`(시/도)만 사용, `TOUR_SIGNGU_NM`(시군구)은 결측 26%·응답 신뢰도 낮아 미사용
- `AGRDE_FLAG_NM`(연령대)은 지역별 지출 격차의 원인 분석에 필요해 사용, `HSHLD_INCOME_DGREE_NM`(소득수준)은 결측 크고 활용 계획 없어 미사용
- 성별(`SEXDSTN_FLAG_CD`)·결혼여부(`MRRG_AT_NM`)·자녀유형(`CHLDRN_TY_NM`)·직업(`OCCP_NM`)은 배경 설명용 보조 변수로 사용
- `COM_ONE_TY`(동행자1순위)만 사용, 2~6순위(`COM_TWO_TY`~`COM_SIX_TY`)는 구조적 결측이라 미사용
- 지출 총액(`TOUR_TOT_CT_VALUE`) 및 6개 항목(`TOUR_LDGMNT_CT_VALUE`, `TOUR_FOOD_CT_VALUE`, `TOUR_TRNSPORT_CT_VALUE`, `TOUR_SHOPNG_CT_VALUE`, `TOUR_ACTVTY_CT_VALUE`, `TOUR_ETC_CT_VALUE`)은 핵심 분석 변수라 전부 사용

## [2] EDA - 결측치 / 이상치 / 손상 레코드 확인 및 파생 변수 생성

### 결측치

#### 결측치 확인

In [4]:
# NaN + '모름'이라는 문자열 응답도 결측으로 하기로 함
rows = []
for col in df.columns:
    nan_count = df[col].isnull().sum()
    unknown_count = df[col].astype(str).str.contains('모름', na=False).sum()
    rows.append({'column': col, 'NaN': nan_count, '모름': unknown_count})

missing_summary = pd.DataFrame(rows).set_index('column')
missing_summary['total_count'] = missing_summary['NaN'] + missing_summary['모름']
missing_summary['total_percent'] = missing_summary['total_count'] / len(df) * 100
missing_summary = missing_summary[missing_summary['total_count'] > 0].sort_values('total_percent', ascending=False)

print(missing_summary)

                          NaN    모름  total_count  total_percent
column                                                         
COM_SIX_TY              14342     0        14342     100.000000
COM_FIVE_TY             14340     0        14340      99.986055
COM_FOUR_TY             14335     0        14335      99.951192
COM_THREE_TY            14212     0        14212      99.093571
COM_TWO_TY              11642     0        11642      81.174174
TOUR_SIGNGU_NM              0  3772         3772      26.300377
HSHLD_INCOME_DGREE_NM       0  3694         3694      25.756519
TOUR_TOT_CT_VALUE           0  2639         2639      18.400502
TOUR_LDGMNT_CT_VALUE        0  2639         2639      18.400502
TOUR_FOOD_CT_VALUE          0  2639         2639      18.400502
TOUR_TRNSPORT_CT_VALUE      0  2639         2639      18.400502
TOUR_SHOPNG_CT_VALUE        0  2639         2639      18.400502
TOUR_ACTVTY_CT_VALUE        0  2639         2639      18.400502
TOUR_ETC_CT_VALUE           0  2639     

##### `TOUR_TOT_CT_VALUE` (여행총비용값): 2,639건 (18.40%)

In [5]:
# 다른 지출 항목으로 교차 추정
cols = ['TOUR_TOT_CT_VALUE', 'TOUR_LDGMNT_CT_VALUE', 'TOUR_FOOD_CT_VALUE', 'TOUR_TRNSPORT_CT_VALUE',
        'TOUR_SHOPNG_CT_VALUE', 'TOUR_ACTVTY_CT_VALUE', 'TOUR_ETC_CT_VALUE']

total_unknown = df['TOUR_TOT_CT_VALUE'] == '모름'
known_total = df[~total_unknown]
unknown_total = df[total_unknown]

check = pd.DataFrame({
    '총지출 있음 & 이 항목 모름': [(known_total[c]=='모름').sum() for c in cols[1:]],
    '총지출 모름 & 이 항목 있음': [(unknown_total[c]!='모름').sum() for c in cols[1:]],
}, index=cols[1:])

print(check)

# 행 제거
before = df.groupby('TOUR_CTPRVN_NM').size()
after = df[df['TOUR_TOT_CT_VALUE'] != '모름'].groupby('TOUR_CTPRVN_NM').size()

change = pd.DataFrame({
    'before': before,
    '제거건수': before - after,
    'after': after,
})
change['제거비율(%)'] = (change['제거건수'] / change['before'] * 100).round(1)
change = change.sort_values('제거비율(%)', ascending=False)
print(change)
print(f'전체: {before.sum()} -> {after.sum()}')

                        총지출 있음 & 이 항목 모름  총지출 모름 & 이 항목 있음
TOUR_LDGMNT_CT_VALUE                   0                 0
TOUR_FOOD_CT_VALUE                     0                 0
TOUR_TRNSPORT_CT_VALUE                 0                 0
TOUR_SHOPNG_CT_VALUE                   0                 0
TOUR_ACTVTY_CT_VALUE                   0                 0
TOUR_ETC_CT_VALUE                      0                 0
                before  제거건수  after  제거비율(%)
TOUR_CTPRVN_NM                              
울산시                223    65    158     29.1
광주시                158    43    115     27.2
경기도               1227   268    959     21.8
경상남도               950   188    762     19.8
충청남도               885   169    716     19.1
대전시                352    67    285     19.0
충청북도               553   105    448     19.0
대구시                332    62    270     18.7
서울시                907   168    739     18.5
부산시               1153   211    942     18.3
제주도               1041   187    854     18.0
인천

1. 최빈값/중앙값 대체
    - "많이 오지만 지출은 낮은 지역"을 가려내는 것이 핵심, 대체값이 실제 지역별 평균 지출을 왜곡할 위험이 큼
2. 다른 지출 항목으로 교차 추정
    - 총 지출과 6개 세부항목의 '모름'을 정/역방향으로 대조했을 때, 총지출과 세부 6항목이 서로 결측
3. 지역/연령대 등 다른 변수 기준 그룹 평균/중앙값으로 대체
    - 지역 평균으로 그 지역의 결측을 메우면 그 메운 값이 다시 지역 평균 계산에 들어가 순환이 됨
    - 지역 간 실제 편차를 인위적으로 줄이는 위험이 있음
4. "모름"을 별도 카테고리로 유지
    - 지출 수준 비교를 위해서 평균 계산을 해야하는데 평균 계산에 숫자를 사용할 수 없어 집계 자체가 불가능
5. [v] 행 삭제
    - 대체할 원천 데이터가 없음, 표본 손실을 감수하더라도 삭제가 유일하게 데이터를 왜곡하지 않는 방법

##### `TOUR_COM_NMPR_NM` (동행자인원): 453건 (3.16%)

1. 최빈값 대체
    - 실제로 응답하지 않은 동행 인원을 임의로 만들어내는 셈이므로, "홀로"/"단체" 파생변수의 신뢰성이 훼손됨
2. 별도 카테고리 유지
    - 전체의 3.2%에 불과해 별도 그룹으로 분리할 실익이 없음
3. [v] 행 삭제
    - 비중이 작고, "홀로"/"단체" 여행 스타일을 판별하는 핵심 컬럼이므로 데이터 신뢰성 확보를 우선시함

##### `COM_ONE_TY` (동행자1유형): 135건 (0.94%)

1. 행 삭제
    - 보조 변수, 이 컬럼 하나 때문에 핵심 변수가 멀쩡한 행까지 지우는 건 손실 대비 실익이 적음
2. 최빈값으로 대체
    - 응답하지 않은 사람들의 동행 유형을 임의로 채워 보조 변수의 분포를 왜곡할 수 있음
3. [v] 별도 카테고리 "미상"로 유지
    - 핵심 지출, 지역 분석에는 영향이 없는 보조 변수이므로 응답하지 않음 자체를 하나의 정보로 남김

##### `TOUR_PD_VALUE` (여행기간): 144건 (1.00%)

1. 행 삭제
    - 핵심 지출, 지역 분석에는 영향이 없는 보조 변수, 다른 분석에 쓸 수 있는 데이터까지 손실됨
2. [v] 행 유지, 파생변수 생성 시점에만 결측 처리
    - 여행일수 파생에서만 NaN으로 남기고 지역별 지출 분석 등 다른 용도에서는 그대로 사용 

#### 결측치 처리 기준
- 결측 처리X
    - 미사용 컬럼: `TOUR_SIGNGU_NM`, `HSHLD_INCOME_DGREE_NM`, `COM_TWO_TY` ~ `COM_SIX_TY`
- 결측 처리O
    - `TOUR_TOT_CT_VALUE` (18.40%): 행 삭제
    - `TOUR_COM_NMPR_NM` (3.16%): 행 삭제
    - `COME_ONE_TY` (0.94%): 별도 카테고리 "미상"으로 변경
    - `TOUR_PD_VALUE` (1.00%): 행 유지, 파생변수 생성 시점에만 결측 처리